# imports

In [30]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [31]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [32]:
ASSET = 'BTC'
INTERVAL = '1h'

In [33]:
query = f"""
    SELECT * 
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

# dataframe

In [34]:
df = conn.execute(query).df()

# datetime index 

In [35]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

In [36]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,BTC,Crypto,Bybit,1h,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,...,0.024733,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5,6.380255e+07,NaN,NaN
2020-04-02 18:00:00,BTC,Crypto,Bybit,1h,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,...,-0.042477,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5,3.068316e+07,NaN,NaN
2020-04-02 19:00:00,BTC,Crypto,Bybit,1h,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,...,0.001399,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5,8.660943e+06,NaN,NaN


# dropping cols and cleaning

In [37]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']

In [38]:
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 47


In [39]:
df.head(5)

,open,high,low,close,volume,daily_volatility,sma_7,sma_30,rsi_14,macd,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,6790.000000,6524.483333,85.656499,130.772825,...,0.024733,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5,6.380255e+07,NaN,NaN
2020-04-02 18:00:00,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,6805.857143,6543.566667,58.856604,123.436911,...,-0.042477,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5,3.068316e+07,NaN,NaN
2020-04-02 19:00:00,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,6826.571429,6563.550000,59.298990,117.040547,...,0.001399,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5,8.660943e+06,NaN,NaN
2020-04-02 20:00:00,6796.5,6799.0,6644.0,6738.0,1411.383,155.0,6840.500000,6580.166667,55.352126,106.028688,...,-0.008645,0.023004,0.606452,6796.5,1274.324,6863.5,6723.0,9.509899e+06,NaN,NaN
2020-04-02 21:00:00,6738.0,6843.0,6711.5,6825.5,1056.962,131.5,6847.642857,6600.233333,59.675391,103.172917,...,0.012902,0.019266,0.866920,6738.0,1411.383,6799.0,6644.0,7.214294e+06,NaN,NaN


# check with new crypto-cols

In [40]:
new_cols = ['turnover', 'open_interest', 'funding_rate']
available_new = [c for c in new_cols if c in df.columns]
missing_new = [c for c in new_cols if c not in df.columns]

print(f"Columns available: {available_new}")
print(f"Columns MISSING:    {missing_new}")

Columns available: ['turnover', 'open_interest', 'funding_rate']
Columns MISSING:    []


In [41]:
for c in available_new:
    coverage = (1 - df[c].isna().mean()) * 100
    print(f"  {c}: {coverage:.1f}% non-null ({df[c].notna().sum():,} / {len(df):,} rows)")

  turnover: 100.0% non-null (53,638 / 53,638 rows)
  open_interest: 95.1% non-null (51,031 / 53,638 rows)
  funding_rate: 3.0% non-null (1,599 / 53,638 rows)


# Drop funding_rate — only ~3% coverage, not usable for ML

In [42]:
if 'funding_rate' in df.columns:
    df = df.drop(columns=['funding_rate'])
    print("Dropped funding_rate — not suitable for model training")
else:
    print("funding_rate not in columns, nothing to drop")

Dropped funding_rate — not suitable for model training


# new feature OI , eda found oi pct_change spikes predict 24× baseline next-hour returns

In [43]:
if 'open_interest' in df.columns:
    df['oi_change_1p']  = df['open_interest'].pct_change(1)
    df['oi_change_5p']  = df['open_interest'].pct_change(5)
    df['oi_change_20p'] = df['open_interest'].pct_change(20)
        
    oi_roll_mean = df['open_interest'].rolling(50).mean()
    oi_roll_std  = df['open_interest'].rolling(50).std()
    df['oi_zscore_50'] = (df['open_interest'] - oi_roll_mean) / oi_roll_std
    
    df = df.drop(columns=['open_interest'])
        
    print("OI features eng: oi_change_1p, oi_change_5p, oi_change_20p, oi_zscore_50")
    print(f"  oi_change_1p  NaN: {df['oi_change_1p'].isna().sum()} rows")
    print(f"  oi_change_5p  NaN: {df['oi_change_5p'].isna().sum()} rows")
    print(f"  oi_change_20p NaN: {df['oi_change_20p'].isna().sum()} rows")
    print(f"  oi_zscore_50  NaN: {df['oi_zscore_50'].isna().sum()} rows")
else:
    print("open_interest not available — skipping OI features")


OI features eng: oi_change_1p, oi_change_5p, oi_change_20p, oi_zscore_50
  oi_change_1p  NaN: 2608 rows
  oi_change_5p  NaN: 2612 rows
  oi_change_20p NaN: 2627 rows
  oi_zscore_50  NaN: 2967 rows


# new features Turnover

In [44]:
if 'turnover' in df.columns:
    df['turnover_ratio'] = df['turnover'] / df['volume']
    
    df['turnover_change_1p'] = df['turnover'].pct_change(1)
    df['turnover_change_5p'] = df['turnover'].pct_change(5)
    
    tv_roll_mean = df['turnover_ratio'].rolling(50).mean()
    tv_roll_std  = df['turnover_ratio'].rolling(50).std()
    df['turnover_ratio_zscore_50'] = (df['turnover_ratio'] - tv_roll_mean) / tv_roll_std
    
    df = df.drop(columns=['turnover'])
    
    print("Turnover features eng: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50")
else:
    print("turnover not available — skipping turnover features")

Turnover features eng: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50


# create target

In [46]:
df['target_1h'] = df['close'].pct_change().shift(-1)

# drop nans from target last row has no next period

In [47]:
df = df.dropna(subset=['target_1h'])

In [48]:
df[['close', 'target_1h']].tail()

,close,target_1h
date,,
2026-05-16 09:00:00,78040.0,-0.001417
2026-05-16 10:00:00,77929.4,0.001433
2026-05-16 11:00:00,78041.1,-0.000869
2026-05-16 12:00:00,77973.3,0.000263
2026-05-16 13:00:00,77993.8,-0.000881


# Drop Remaining NaN Rows & Check Shape

In [49]:
df = df.dropna()
print(f"Data shape after dropping NaN: {df.shape}")
print(f"Total features: {len(df.columns) - 2}  (+ target_1h + target_direction)")

Data shape after dropping NaN: (50670, 53)
Total features: 51  (+ target_1h + target_direction)


In [50]:
df['target_direction'] = (df['target_1h'] > 0).astype(int)
balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance:")
print(balance)

Class Balance:
target_direction
1    50.590093
0    49.409907
Name: proportion, dtype: float64


# Convert to Stationary Percentages
# Same transform as v1 — raw prices become distances/ratios

# 01 — Moving Averages → Distance from Price (%)

In [51]:
df['sma_7_dist']   = df['close'] / df['sma_7']   - 1
df['sma_30_dist']  = df['close'] / df['sma_30']  - 1
df['sma_50_dist']  = df['close'] / df['sma_50']  - 1
df['sma_100_dist'] = df['close'] / df['sma_100'] - 1
df['sma_200_dist'] = df['close'] / df['sma_200'] - 1
df['ema_12_dist']  = df['close'] / df['ema_12']  - 1
df['ema_26_dist']  = df['close'] / df['ema_26']  - 1
df['ema_50_dist']  = df['close'] / df['ema_50']  - 1
df['ema_200_dist'] = df['close'] / df['ema_200'] - 1
df['vwap_dist']    = df['close'] / df['vwap']    - 1

# 02 — MACD & Volatility → % of Price

In [52]:
df['macd_pct']      = df['macd']            / df['close']
df['macd_sig_pct']  = df['macd_signal']     / df['close']
df['macd_hist_pct'] = df['macd_histogram']  / df['close']
df['atr_pct']       = df['atr_14']          / df['close']
df['volatility_pct']= df['daily_volatility'] / df['close']

# Drop Non-Stationary Columns

In [53]:
non_stationary_cols = [
    'open', 'high', 'low', 'close', 
    'sma_7', 'sma_30', 'sma_50', 'sma_100', 'sma_200',
    'ema_12', 'ema_26', 'ema_50', 'ema_200', 
    'vwap', 'prev_close', 'prev_high', 'prev_low',
    'macd', 'macd_signal', 'macd_histogram', 
    'atr_14', 'daily_volatility',            
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 
    'volume', 'prev_volume', 'volume_sma_20', 
    'obv' 
]
df = df.drop(columns=non_stationary_cols, errors='ignore')

# Final Cleanup

In [54]:
df = df.dropna()
print(f"Final data shape for ML: {df.shape}")

all_features = [c for c in df.columns if c not in ['target_1h', 'target_direction']]
print(f"\nFeatures ({len(all_features)}):")
for i, f in enumerate(all_features):
    print(f"  {i+1:2d}. {f}")

Final data shape for ML: (50670, 39)

Features (37):
   1. rsi_14
   2. roc_10
   3. roc_20
   4. stoch_k
   5. stoch_d
   6. bb_percentage
   7. volume_ratio
   8. returns_1p
   9. returns_5p
  10. returns_10p
  11. returns_20p
  12. log_returns
  13. hl_ratio
  14. close_position
  15. oi_change_1p
  16. oi_change_5p
  17. oi_change_20p
  18. oi_zscore_50
  19. turnover_ratio
  20. turnover_change_1p
  21. turnover_change_5p
  22. turnover_ratio_zscore_50
  23. sma_7_dist
  24. sma_30_dist
  25. sma_50_dist
  26. sma_100_dist
  27. sma_200_dist
  28. ema_12_dist
  29. ema_26_dist
  30. ema_50_dist
  31. ema_200_dist
  32. vwap_dist
  33. macd_pct
  34. macd_sig_pct
  35. macd_hist_pct
  36. atr_pct
  37. volatility_pct


# train/ test split 80/20 chronological, no shuffle

In [55]:
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

In [56]:
print(f"Training: {train_df.index.min()} → {train_df.index.max()}  ({len(train_df):,} rows)")
print(f"Testing:  {test_df.index.min()} → {test_df.index.max()}  ({len(test_df):,} rows)")

Training: 2020-08-04 08:00:00 → 2025-03-20 07:00:00  (40,536 rows)
Testing:  2025-03-20 08:00:00 → 2026-05-16 13:00:00  (10,134 rows)


# Feature Set (exclude targets)

In [57]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
print(f"{len(features)} features for model input")

37 features for model input


# StandardScaler (fit on train, apply to test)

In [58]:
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features]  = scaler.transform(test_df[features])

# Save to Parquet

In [59]:
train_df.to_parquet('../../data/processed/train_btc_1h_v2.parquet')
test_df.to_parquet('../../data/processed/test_btc_1h_v2.parquet')
print("Saved: train_btc_1h_v2.parquet + test_btc_1h_v2.parquet")
print("(v2 = 29 original features + OI & turnover features)")

Saved: train_btc_1h_v2.parquet + test_btc_1h_v2.parquet
(v2 = 29 original features + OI & turnover features)


In [60]:
conn.close()